In [ ]:
!pip install timm -q

In [ ]:
import os, sys, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score

sys.path.append('/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/')

from src.config_2d import Config2D
from src.dataset_2d import SpectrogramDataset, preprocess_spectrograms
from src.model_2d import build_model_2d

In [ ]:
cfg = Config2D()
cfg.spectrogram_dir = '/kaggle/input/competitions/hms-harmful-brain-activity-classification/train_spectrograms/'
cfg.metadata_csv    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
cfg.spec_cache_dir  = '/tmp/hms_spec_cache'

SEED = cfg.seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

cfg.lr = 1e-4
print(cfg.as_dict())

In [ ]:
# Convert parquet → npy once; subsequent runs skip existing files.
preprocess_spectrograms(cfg.spectrogram_dir, cfg.spec_cache_dir, n_jobs=-1)

In [ ]:
train_ds = SpectrogramDataset(cfg.metadata_csv, cfg.spec_cache_dir, cfg, split='train')
val_ds   = SpectrogramDataset(cfg.metadata_csv, cfg.spec_cache_dir, cfg, split='val')

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

# sanity-check batch shapes
batch = next(iter(train_dl))
print(f"image shape   : {batch['image'].shape}")
print(f"soft_label    : {batch['soft_label'].shape}")
print(f"label sample  : {batch['label'][:4]}")

In [ ]:
model = build_model_2d(cfg).to(cfg.device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone      : {cfg.backbone}")
print(f"Parameters    — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
DEVICE    = cfg.device
USE_AMP   = (DEVICE == 'cuda')
NUM_EPOCHS = cfg.num_epochs
patience   = cfg.patience

# ── Augmentation helpers ──────────────────────────────────────────────────────
def freq_mask(img, max_f=20):
    # img shape: (4, 100, 300)
    f  = random.randint(0, max_f)
    f0 = random.randint(0, 100 - f)
    img[:, f0:f0+f, :] = 0
    return img

def time_mask(img, max_t=30):
    t  = random.randint(0, max_t)
    t0 = random.randint(0, 300 - t)
    img[:, :, t0:t0+t] = 0
    return img

def mixup_batch(images, labels, alpha=2.0):
    lam         = np.random.beta(alpha, alpha)
    idx         = torch.randperm(images.size(0))
    mixed_images = lam * images + (1 - lam) * images[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_images, mixed_labels

criterion = nn.KLDivLoss(reduction='batchmean')
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_f1, best_epoch, history = 0.0, 0, []
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    t0         = time.time()
    train_loss = 0.0
    for batch in train_dl:
        images = batch['image']
        soft_y = batch['soft_label'].to(DEVICE)

        # freq/time mask per sample (50% probability each)
        for i in range(images.size(0)):
            img_np = images[i].cpu().numpy()
            if random.random() < 0.5:
                img_np = freq_mask(img_np)
            if random.random() < 0.5:
                img_np = time_mask(img_np)
            images[i] = torch.from_numpy(img_np)
        images = images.to(DEVICE)
        # MixUp
        images, soft_y = mixup_batch(images, soft_y, alpha=2.0)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss   = F.kl_div(F.log_softmax(logits, dim=1), soft_y, reduction='batchmean')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # ── validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss_total = 0.0
    val_kl_total   = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_dl:
            images  = batch['image'].to(DEVICE)
            soft_np = batch['soft_label'].numpy()
            labels  = batch['label']
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(images)
            val_loss_total += F.kl_div(
                F.log_softmax(logits, dim=1),
                batch['soft_label'].to(DEVICE),
                reduction='batchmean',
            ).item()
            prob = F.softmax(logits, dim=1).cpu().numpy()
            kl   = (np.clip(soft_np, 1e-7, 1) *
                    np.log(np.clip(soft_np, 1e-7, 1) / np.clip(prob, 1e-7, 1))).sum(axis=1).mean()
            val_kl_total += kl
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())

    avg_train = train_loss     / len(train_dl)
    avg_val   = val_loss_total / len(val_dl)
    avg_kl    = val_kl_total   / len(val_dl)
    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    elapsed   = time.time() - t0

    history.append({
        'epoch'     : epoch,
        'train_loss': avg_train,
        'val_loss'  : avg_val,
        'val_kl'    : avg_kl,
        'macro_f1'  : macro_f1,
    })
    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_kl:.4f} | val_loss {avg_val:.4f} | "
          f"macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    if macro_f1 > best_f1:
        best_f1    = macro_f1
        best_epoch = epoch
        wait       = 0
        torch.save(model.state_dict(), 'best_cnn2d_v3.pt')
        print(f"  \u2713 saved best_cnn2d_v3.pt (macro_f1={best_f1:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} (patience={patience})")
            break

print(f"\nTraining complete. Best macro_f1={best_f1:.4f} at epoch {best_epoch}")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
print(f"Best epoch    : {best_epoch}")
print(f"Best macro_F1 : {best_f1:.4f}")
best_kl = min(h['val_kl'] for h in history)
print(f"Best val KL   : {best_kl:.4f}  "
      f"(epoch {next(h['epoch'] for h in history if h['val_kl'] == best_kl)})")

# ── Per-epoch log (copy-paste friendly for ablation table) ───────────────────
print(f"\n{'epoch':>5}  {'train_kl':>9}  {'val_loss':>9}  {'val_kl':>7}  {'macro_f1':>9}")
for h in history:
    marker = ' ←' if h['epoch'] == best_epoch else ''
    print(f"{h['epoch']:>5}  {h['train_loss']:>9.4f}  "
          f"{h['val_loss']:>9.4f}  {h['val_kl']:>7.4f}  "
          f"{h['macro_f1']:>9.4f}{marker}")

# ── Plots ─────────────────────────────────────────────────────────────────────
epochs   = [h['epoch']     for h in history]
train_kl = [h['train_loss'] for h in history]
val_kl   = [h['val_kl']    for h in history]
macro_f1 = [h['macro_f1']  for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs, train_kl, label='train KL')
ax1.plot(epochs, val_kl,   label='val KL')
ax1.axvline(best_epoch, color='red', linestyle='--', label=f'best epoch={best_epoch}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('KL Divergence')
ax1.set_title('cnn2d-v3 (augmentation) — KL Divergence'); ax1.legend()

ax2.plot(epochs, macro_f1, color='green')
ax2.axvline(best_epoch, color='red', linestyle='--', label=f'best={best_f1:.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('cnn2d-v3 (augmentation) — Validation Macro F1'); ax2.legend()

plt.tight_layout()
plt.savefig('cnn2d_v3_curves.png', dpi=150)
plt.show()